# Beat Detector learning mode

Adapts the module's neural beat model to your own material: audio with a beat grid aligned to it.
The top layers are fine-tuned, pulled towards the original weights so the model keeps what it
knew, and the last 20% of every recording is held back to check that the adapted model really is
better. The result is an RTNeural JSON file you load in Rack.

A GPU (*Runtime → Change runtime type*) helps, but fine-tuning only takes a few minutes on CPU.

**Recordings** go in a Google Drive folder, `MyDrive/BeatDetector/recordings` by default. Each
audio file (WAV, FLAC, AIFF, OGG or MP3) needs a `.beats` file with the same name beside it: one
line per beat, the time in seconds, then optionally its position in the bar (1 = downbeat):

```
0.512	1
1.024	2
1.536	3
```

About four minutes of material is enough to start: in tests, a model that had never heard a
waltz caught up with one trained on every waltz after four minutes of them.

In [ ]:
REF = 'main'  # branch or tag of github.com/un1tz3r0/vcvtempodetector
!pip install -q "beatref @ git+https://github.com/un1tz3r0/vcvtempodetector@{REF}#subdirectory=python"
!wget -q -O beat.json https://raw.githubusercontent.com/un1tz3r0/vcvtempodetector/{REF}/plugin/res/models/beat.json
import torch
print('GPU' if torch.cuda.is_available() else 'CPU only (fine for fine-tuning)')

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/BeatDetector'
RECORDINGS = f'{ROOT}/recordings'
BASE = 'beat.json'                   # or a model of your own, e.g. f'{ROOT}/models/house.json'
OUT = f'{ROOT}/models/adapted.json'
os.makedirs(os.path.dirname(OUT), exist_ok=True)

### Optional: replay data

Mixing some of the original kind of training data into the batches keeps the adapted model good
at everything else, at the cost of a little less adaptation. Point `HF_DATASET` at a Hugging Face
dataset repo of audio with `.beats` files. For a private repo, add an `HF_TOKEN` secret (the key
icon in the left sidebar) and give this notebook access to it.

In [ ]:
HF_DATASET = ''  # e.g. 'your-name/beat-corpus'
REPLAY = []
if HF_DATASET:
    from google.colab import userdata
    from huggingface_hub import snapshot_download
    REPLAY = [snapshot_download(HF_DATASET, repo_type='dataset', token=userdata.get('HF_TOKEN'), local_dir='/content/replay')]

### Fine-tune

`layers` is how many layers from the top are adapted (0 = all of them: fits your material best,
but forgets the most); `l2sp` is how hard they are pulled back towards the base model. The model is written only if it beats the base model on
the held-back audio; its latency is then recalibrated, and the tempo range and meter of your
recordings are stored in it as suggested settings.

In [ ]:
from beatref.finetune import Tune, finetune
tune = Tune(layers=3, l2sp=1e-4, replay=0.5 if REPLAY else 0.0)
finetune(BASE, [RECORDINGS], OUT, tune, replay=REPLAY, cache='/content/cache')

### Compare

Tracks every recording with the base and the adapted model. Most of each recording was trained
on, so this is a sanity check rather than a fair test: score new recordings for that.

In [ ]:
from beatref.config import Config
from beatref.evaluate import evaluate
for model in (BASE, OUT):
    print(model)
    evaluate(RECORDINGS, Config(model=model))

### Use it in Rack

Copy `adapted.json` from your Drive, then right-click the module: *Beat activation → Load
RTNeural JSON…* (or put it in the *Learning mode → Model library* folder). The model is saved in
the patch. *Learning mode → Apply the model's suggested settings* sets the tempo range and
Beats/bar to match your recordings; for material outside 90–180 BPM this matters as much as the
adaptation.